# 4-bob. 4-BOB. XAVFSIZLIK MA'LUMOTLARINI TAYYORLASH
## Kengaytirilgan amaliy ish: belgilash funksiyalari bilan model o'qitish

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/04-bob/4-1_belgilash-funksiyalari-bilan-model-oqitish.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 4-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dastur 4400 ta sun'iy URL manzil yaratadi, ulardan 400 tasi beshta turdagi fishing havolalari. Haqiqiy belgilar faqat baholash uchun ishlatiladi. Yettita belgilash funksiyasi yoziladi, ulardan oxirgisi ataylab "shovqinli": u https protokolidagi barcha havolalarni zararsiz deb hisoblaydi. Funksiyalar ko'pchilik ovozi bilan birlashtiriladi va olingan kuchsiz belgilarda URL belgilarining n-gramlariga asoslangan logistik regressiya o'qitiladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

rng = np.random.default_rng(4)

In [2]:
# 1. Sun'iy URL manzillar: haqiqiy belgilar faqat baholash uchun ishlatiladi
yaxshi_dom = ["uzcard.uz", "kun.uz", "my.gov.uz", "tsue.uz", "click.uz", "payme.uz", "google.com", "lex.uz"]
yol = ["/", "/news", "/login", "/uz/page", "/profile", "/pay?id=", "/docs/"]
def oddiy():
    return f"https://{'www.' if rng.random() < .5 else ''}{rng.choice(yaxshi_dom)}{rng.choice(yol)}{rng.integers(1, 9999)}"
def fishing():
    tur = rng.integers(0, 5)
    brend = rng.choice(["uzcard", "click", "payme", "mygov", "humo"])
    if tur == 0: return f"http://{rng.integers(11, 223)}.{rng.integers(0, 255)}.{rng.integers(0, 255)}.{rng.integers(1, 254)}/{brend}/login"
    if tur == 1: return f"http://{brend}-bonus.{rng.choice(['top', 'xyz', 'click', 'site'])}/olish"
    if tur == 2: return f"https://{brend}.uz.verify-{rng.integers(100, 999)}.com/kirish"
    if tur == 3: return f"http://secure-{brend}-uz.{rng.choice(['online', 'info', 'live'])}/karta?raqam="
    return f"https://bit.ly/{''.join(rng.choice(list('abcdefgh123'), 6))}"     # qisqartirilgan havola
urls = [oddiy() for _ in range(4000)] + [fishing() for _ in range(400)]
y = np.r_[np.zeros(4000), np.ones(400)].astype(int)
idx = rng.permutation(len(urls)); urls = [urls[i] for i in idx]; y = y[idx]
X_o, y_o, X_s, y_s = urls[:3000], y[:3000], urls[3000:], y[3000:]

In [3]:
# 2. Belgilash funksiyalari: 1 - fishing, 0 - oddiy, -1 - "bilmayman"
IP = re.compile(r"//\d+\.\d+\.\d+\.\d+")
def bf_ip(u):        return 1 if IP.search(u) else -1
def bf_tld(u):       return 1 if re.search(r"\.(top|xyz)/", u) else -1          # faqat ikkita domen zonasi
def bf_ichki(u):     return 1 if re.search(r"\.uz\.[a-z0-9-]+\.com", u) else -1
def bf_tire(u):      return 1 if u.split("/")[2].count("-") >= 2 else -1
def bf_ishonchli(u): return 0 if u.split("/")[2].removeprefix("www.") in yaxshi_dom else -1
def bf_qisqa(u):     return 1 if "bit.ly/" in u else -1
def bf_https(u):     return 0 if u.startswith("https://") else -1        # shovqinli funksiya
funksiyalar = [bf_ip, bf_tld, bf_ichki, bf_tire, bf_qisqa, bf_ishonchli, bf_https]

L = np.array([[f(u) for f in funksiyalar] for u in X_o])
print("Funksiya          qamrov  aniqlik (haqiqiy belgilar bilan tekshirildi)")
for j, f in enumerate(funksiyalar):
    ishladi = L[:, j] != -1
    print(f"{f.__name__:<16} {ishladi.mean():6.1%}  {(L[ishladi, j] == y_o[ishladi]).mean():6.1%}")

Funksiya          qamrov  aniqlik (haqiqiy belgilar bilan tekshirildi)
bf_ip              1.7%  100.0%
bf_tld             0.9%  100.0%
bf_ichki           1.9%  100.0%
bf_tire            1.6%  100.0%
bf_qisqa           1.7%  100.0%
bf_ishonchli      91.4%  100.0%
bf_https          94.9%   96.2%


In [4]:
# 3. Ko'pchilik ovozi bilan kuchsiz belgilar; teng ovoz yoki hech biri ishlamasa - yozuv tashlab yuboriladi
def kuchsiz_belgilar(L):
    ovoz1, ovoz0 = (L == 1).sum(1), (L == 0).sum(1)
    return np.where(ovoz1 > ovoz0, 1, np.where(ovoz0 > ovoz1, 0, -1))

vek = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(X_o)
def baholash(nom, X, y_):
    m = LogisticRegression(max_iter=2000, class_weight="balanced").fit(vek.transform(X), y_)
    p = m.predict(vek.transform(X_s))
    print(f"{nom:<44} aniqlik={precision_score(y_s, p):.3f}  to'liqlik={recall_score(y_s, p):.3f}")

print()
for nom, ustunlar in [("barcha funksiyalar", list(range(len(funksiyalar)))),
                      ("bf_https olib tashlangan", list(range(len(funksiyalar) - 1)))]:
    kb = kuchsiz_belgilar(L[:, ustunlar]); tanlandi = kb != -1
    print(f"Kuchsiz belgilar ({nom}): {tanlandi.mean():.1%} yozuv, aniqligi {(kb[tanlandi] == y_o[tanlandi]).mean():.1%}")
    baholash(f"  model, kuchsiz belgilar ({nom})", [X_o[i] for i in np.where(tanlandi)[0]], kb[tanlandi])


Kuchsiz belgilar (barcha funksiyalar): 95.5% yozuv, aniqligi 100.0%


  model, kuchsiz belgilar (barcha funksiyalar) aniqlik=1.000  to'liqlik=0.539
Kuchsiz belgilar (bf_https olib tashlangan): 99.1% yozuv, aniqligi 100.0%


  model, kuchsiz belgilar (bf_https olib tashlangan) aniqlik=1.000  to'liqlik=1.000


In [5]:
# 4. Taqqoslash uchun: funksiyalarning o'zi va haqiqiy belgilarda o'qitilgan model
qoida = np.array([int(any(f(u) == 1 for f in funksiyalar)) for u in X_s])
print(f"{'Faqat belgilash funksiyalari':<44} aniqlik={precision_score(y_s, qoida):.3f}  to'liqlik={recall_score(y_s, qoida):.3f}")
baholash("Model, haqiqiy belgilar (3000 ta qo'lda)", X_o, y_o)

Faqat belgilash funksiyalari                 aniqlik=1.000  to'liqlik=0.915


Model, haqiqiy belgilar (3000 ta qo'lda)     aniqlik=1.000  to'liqlik=1.000


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Funksiya          qamrov  aniqlik (haqiqiy belgilar bilan tekshirildi)
bf_ip              1.7%  100.0%
bf_tld             0.9%  100.0%
bf_ichki           1.9%  100.0%
bf_tire            1.6%  100.0%
bf_qisqa           1.7%  100.0%
bf_ishonchli      91.4%  100.0%
bf_https          94.9%   96.2%

Kuchsiz belgilar (barcha funksiyalar): 95.5% yozuv, aniqligi 100.0%
  model, kuchsiz belgilar (barcha funksiyalar) aniqlik=1.000  to'liqlik=0.539
Kuchsiz belgilar (bf_https olib tashlangan): 99.1% yozuv, aniqligi 100.0%
  model, kuchsiz belgilar (bf_https olib tashlangan) aniqlik=1.000  to'liqlik=1.000
Faqat belgilash funksiyalari                 aniqlik=1.000  to'liqlik=0.915
Model, haqiqiy belgilar (3000 ta qo'lda)     aniqlik=1.000  to'liqlik=1.000
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 4-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.